# Multi-Agent Shared Memory

> *A shared memory layer for collaborating agents, because teamwork requires common ground.*

Think of a team of chefs in a kitchen. Each chef has their own cutting board (private workspace), but they all share one recipe book on the counter. Any chef can read the book, and any chef can add notes. Without that shared book, every chef works in isolation.

Multi-agent shared memory works the same way for AI agents.

When a single agent handles a task, its memory is straightforward. One store, one reader, one writer. But modern agent systems use multiple specialized agents working together. A researcher gathers information. A coder writes code. A reviewer checks quality. These agents need to share information.

The researcher's findings must reach the coder. The coder's output must reach the reviewer. Without shared memory, agents pass everything through conversation turns. That's slow and lossy (meaning information gets lost along the way).

Multi-agent shared memory provides the data layer for agent collaboration. The concept builds on **blackboard systems** from classical AI (Nii, 1986). A blackboard system is a shared workspace where specialist agents post partial results. Other specialists read and refine those results. Modern versions use a shared key-value store or dictionary as the blackboard.

**Namespaces** (logical partitions like `research/`, `code/`, `review/`) organize the memory by domain. **Access control** rules govern which agents can read or write each partition. **Conflict resolution** handles the case when two agents update the same entry at the same time.

In this notebook, you'll build a complete shared memory system from scratch. You'll create three agents (researcher, coder, reviewer) that collaborate on a programming task through shared memory. You'll see how namespaces, permissions, and conflict resolution work together.

**By the end you'll understand:**
- How to design a shared memory pool with namespace-based organization.
- How access control prevents agents from overwriting each other's work.
- How conflict resolution strategies handle concurrent updates.
- How the blackboard pattern coordinates multi-agent collaboration.

## Key Concepts

- **Shared memory store**: A centralized data store that multiple agents read from and write to. In this notebook, we use a Python dictionary. In production, you might use Redis or a database. The store must support concurrent access (multiple agents reading and writing at the same time) without corrupting data.

- **Namespace**: A logical partition within shared memory. For example, `research/`, `code/`, and `review/` are three namespaces. Namespaces organize information by domain. They also reduce conflicts by keeping each agent's writes in their own area.

- **Access control**: Rules that determine which agents can read or write each namespace. For example, the researcher agent might have write access to `research/` but only read access to `code/`. This prevents accidental overwrites.

- **Private scratchpad**: An agent's internal notes, invisible to other agents. Think of it as a personal notebook versus the shared whiteboard. The scratchpad holds drafts and reasoning. Only finished results go into shared memory.

- **Blackboard system**: A classic AI architecture where agents post partial solutions to a shared "blackboard." Other agents read, refine, or extend those solutions. The blackboard is both a communication channel and a collective workspace.

- **Conflict resolution**: A strategy for handling simultaneous updates to the same entry. **Last-write-wins** keeps the most recent write and discards the earlier one. **Version checking** (also called optimistic locking) detects conflicts by tracking version numbers and rejects stale writes.

- **Thread safety**: Protecting shared data from corruption when multiple threads (parallel execution paths) access it at the same time. We use a **lock** (a mechanism that lets only one thread access a resource at a time) to ensure safe concurrent access.

## Architecture

The diagram below shows how private agent memory connects to shared memory through an access control layer.

<p align="center">
  <img src="../../images/diagrams/22_multi_agent_shared_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart TD
    subgraph "Agent A (Researcher)"
        PA[Private<br/>Scratchpad A]
    end

    subgraph "Agent B (Coder)"
        PB[Private<br/>Scratchpad B]
    end

    subgraph "Agent C (Reviewer)"
        PC[Private<br/>Scratchpad C]
    end

    PA --> ACL[Access Control<br/>Layer]
    PB --> ACL
    PC --> ACL

    subgraph "Shared Memory Pool"
        NS1["research/<br/>findings & sources"]
        NS2["code/<br/>artifacts & specs"]
        NS3["review/<br/>feedback & issues"]
    end

    ACL <-->|read/write| NS1
    ACL <-->|read/write| NS2
    ACL <-->|read/write| NS3

    NS1 --> CR[Conflict<br/>Resolution]
    NS2 --> CR
    NS3 --> CR
    CR --> BB[Blackboard<br/>partial solutions]

    style ACL fill:#845ef7,color:#fff
    style CR fill:#ff6b6b,color:#fff
    style BB fill:#51cf66,color:#fff
```

</details>

**Data flow:** Each agent keeps a private scratchpad for internal reasoning. Other agents can't see it. When an agent shares information or reads others' contributions, the request goes through the Access Control Layer. This layer checks permissions and routes reads/writes to the correct namespace. The Shared Memory Pool is split into namespaces by domain (research, code, review). When two agents write to the same key, the Conflict Resolution module picks a winner. The Blackboard provides a unified view of all shared entries.

## Setup

Install dependencies and configure API access. You'll need an `OPENAI_API_KEY` environment variable set in a `.env` file.

In [ ]:
%pip install -q openai python-dotenv

Import the OpenAI SDK and standard library helpers. The API key loads from a `.env` file.

In [ ]:
import os
import time
import threading
from dataclasses import dataclass, field
from enum import Enum

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads OPENAI_API_KEY from .env

client = OpenAI()  # uses OPENAI_API_KEY from environment

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

MODEL = "gpt-4o-mini"  # cost-effective model for demos

## Implementation

We'll build four components:

1. **`Permission`** and **`MemoryEntry`**: Data types for access control and stored entries.
2. **`AccessControl`**: Manages per-agent, per-namespace read/write permissions.
3. **`SharedMemoryPool`**: The central store with thread-safe reads, writes, and conflict resolution.
4. **`Agent`**: An LLM-powered agent with a private scratchpad and shared memory access.

### Data Types

`Permission` defines three access levels: read-only, write-only, and read-write. `MemoryEntry` holds one item in shared memory. It tracks who wrote it and its version number.

In [ ]:
class Permission(Enum):
    """Access level for an agent in a namespace."""
    READ = "read"
    WRITE = "write"
    READ_WRITE = "read_write"


@dataclass
class MemoryEntry:
    """A single entry in shared memory."""
    namespace: str       # e.g., "research", "code"
    key: str             # identifier within the namespace
    value: str           # the stored content
    author: str          # agent ID that wrote this
    version: int = 1     # increments on each update
    timestamp: float = field(default_factory=time.time)

    def __repr__(self) -> str:
        preview = self.value[:60] + "..." if len(self.value) > 60 else self.value
        return f"MemoryEntry({self.namespace}/{self.key} v{self.version} by {self.author})"

### Access Control Layer

The `AccessControl` class maps each agent to its allowed operations on each namespace. Every read and write passes through this layer before touching shared memory.

In [ ]:
class AccessControl:
    """Manages per-agent, per-namespace permissions."""

    def __init__(self):
        self._permissions: dict[str, dict[str, Permission]] = {}

    def grant(self, agent_id: str, namespace: str, permission: Permission) -> None:
        """Give an agent a specific permission on a namespace."""
        self._permissions.setdefault(agent_id, {})[namespace] = permission

    def can_read(self, agent_id: str, namespace: str) -> bool:
        """Check if an agent can read from a namespace."""
        perm = self._permissions.get(agent_id, {}).get(namespace)
        return perm in (Permission.READ, Permission.READ_WRITE)

    def can_write(self, agent_id: str, namespace: str) -> bool:
        """Check if an agent can write to a namespace."""
        perm = self._permissions.get(agent_id, {}).get(namespace)
        return perm in (Permission.WRITE, Permission.READ_WRITE)

    def describe(self) -> str:
        """Return a human-readable summary of all permissions."""
        lines = []
        for agent_id, ns_perms in sorted(self._permissions.items()):
            for ns, perm in sorted(ns_perms.items()):
                lines.append(f"  {agent_id}: {ns} -> {perm.value}")
        return "\n".join(lines)

### Shared Memory Pool

This is the central "blackboard." It stores all shared entries, enforces permissions, and handles concurrent writes with a threading lock.

The pool supports two conflict resolution strategies:

- **`last_write_wins`**: The most recent write overwrites any previous value. Fast but may lose data.
- **`version_check`**: Each write must provide the current version number. If someone else updated the entry first, the write fails. This is also called optimistic locking.

In [ ]:
class SharedMemoryPool:
    """Thread-safe shared memory with namespaces and conflict resolution."""

    def __init__(self, acl: AccessControl, conflict_strategy: str = "last_write_wins"):
        self.acl = acl
        self._store: dict[str, MemoryEntry] = {}  # full_key -> entry
        self._lock = threading.Lock()
        self.conflict_strategy = conflict_strategy
        self._history: list[dict] = []  # audit log


The `write` method checks permissions first, then applies the conflict resolution strategy. With `version_check`, it rejects writes when the caller's expected version doesn't match the stored version. With `last_write_wins`, the newest value always overwrites.

In [ ]:
def write(
    self,
    agent_id: str,
    namespace: str,
    key: str,
    value: str,
    expected_version: int | None = None,
) -> MemoryEntry:
    """Write an entry to shared memory.

    Checks permissions first, then applies the conflict strategy.
    Pass expected_version with version_check strategy to detect conflicts."""
    if not self.acl.can_write(agent_id, namespace):
        raise PermissionError(
            f"Agent '{agent_id}' cannot write to namespace '{namespace}'"
        )

    full_key = f"{namespace}/{key}"

    with self._lock:
        existing = self._store.get(full_key)

        # Version check: reject writes based on stale version
        if (
            self.conflict_strategy == "version_check"
            and existing
            and expected_version is not None
            and existing.version != expected_version
        ):
            raise ValueError(
                f"Version conflict on '{full_key}': "
                f"expected v{expected_version}, found v{existing.version}"
            )

        new_version = (existing.version + 1) if existing else 1

        entry = MemoryEntry(
            namespace=namespace,
            key=key,
            value=value,
            author=agent_id,
            version=new_version,
        )
        self._store[full_key] = entry

        self._history.append({
            "action": "write",
            "agent": agent_id,
            "key": full_key,
            "version": new_version,
            "timestamp": entry.timestamp,
        })

    return entry

SharedMemoryPool.write = write

Now we add the read methods to `SharedMemoryPool`.
One fetches a specific entry by namespace and key.
The other returns all entries the agent has permission to see.

In [ ]:
    def read(
        self, agent_id: str, namespace: str, key: str | None = None
    ) -> list[MemoryEntry]:
        """Read entries from a namespace.

        If key is provided, return that specific entry.
        If key is None, return all entries in the namespace."""
        if not self.acl.can_read(agent_id, namespace):
            raise PermissionError(
                f"Agent '{agent_id}' cannot read from namespace '{namespace}'"
            )

        with self._lock:
            if key:
                full_key = f"{namespace}/{key}"
                entry = self._store.get(full_key)
                return [entry] if entry else []
            else:
                return [
                    e for e in self._store.values()
                    if e.namespace == namespace
                ]

    def read_all_accessible(
        self, agent_id: str, namespaces: list[str] | None = None
    ) -> list[MemoryEntry]:
        """Read all entries the agent has permission to see."""
        results = []
        with self._lock:
            for entry in self._store.values():
                if namespaces and entry.namespace not in namespaces:
                    continue
                if self.acl.can_read(agent_id, entry.namespace):
                    results.append(entry)
        return results

Finally, we add utility methods for auditing and inspection.
`get_history` returns the full write log.
`summary` formats the current memory contents for display.

In [ ]:
    def get_history(self) -> list[dict]:
        """Return the audit log of all write operations."""
        return list(self._history)

    def summary(self) -> str:
        """Return a formatted summary of everything in shared memory."""
        if not self._store:
            return "(empty)"
        lines = []
        for full_key in sorted(self._store):
            entry = self._store[full_key]
            lines.append(
                f"[{full_key}] (v{entry.version}, by {entry.author}):\n"
                f"  {entry.value[:200]}"
            )
        return "\n\n".join(lines)

### Agent with Private Scratchpad

Each agent wraps an LLM call with two memory layers:

1. **Private scratchpad**: A list of notes visible only to this agent. Good for internal reasoning and drafts.
2. **Shared memory access**: Read and write operations that go through the access control layer.

When an agent runs a task, it gathers context from shared memory, adds its private notes, and sends everything to the LLM.

In [ ]:
class Agent:
    """An LLM-powered agent with private scratchpad and shared memory access."""

    def __init__(
        self,
        agent_id: str,
        role: str,
        system_prompt: str,
        shared_memory: SharedMemoryPool,
        model: str = MODEL,
    ):
        self.agent_id = agent_id
        self.role = role
        self.system_prompt = system_prompt
        self.shared_memory = shared_memory
        self.model = model
        self.scratchpad: list[str] = []  # private, not shared

    def think(self, thought: str) -> None:
        """Add a private note to the scratchpad. Other agents cannot see this."""
        self.scratchpad.append(thought)

    def read_shared(self, namespace: str, key: str | None = None) -> list[MemoryEntry]:
        """Read from shared memory (permission-checked)."""
        return self.shared_memory.read(self.agent_id, namespace, key)

    def write_shared(self, namespace: str, key: str, value: str) -> MemoryEntry:
        """Write to shared memory (permission-checked)."""
        return self.shared_memory.write(self.agent_id, namespace, key, value)

The `run` method is where the agent does its work.
It gathers context from shared memory and the private scratchpad, then sends everything to the LLM.
Notice how it only reads from namespaces the agent has permission to access.

In [ ]:
    def run(self, task: str, context_namespaces: list[str] | None = None) -> str:
        """Execute a task using the LLM with shared memory as context.

        Steps:
        1. Gather readable entries from shared memory.
        2. Include private scratchpad notes.
        3. Call the LLM with the combined context.
        4. Return the response."""
        # Build shared memory context
        shared_entries = self.shared_memory.read_all_accessible(
            self.agent_id, context_namespaces
        )
        shared_context = ""
        if shared_entries:
            shared_context = "## Shared Memory (visible to all agents)\n\n"
            for entry in shared_entries:
                shared_context += (
                    f"**[{entry.namespace}/{entry.key}]** (by {entry.author}):\n"
                    f"{entry.value}\n\n"
                )

        # Build private scratchpad context
        scratchpad_context = ""
        if self.scratchpad:
            scratchpad_context = "## Your Private Scratchpad\n\n"
            for note in self.scratchpad:
                scratchpad_context += f"- {note}\n"
            scratchpad_context += "\n"

        user_message = f"{shared_context}{scratchpad_context}## Current Task\n\n{task}"

        response = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_message},
            ],
            max_tokens=1024,
            temperature=0.7,
        )

        return response.choices[0].message.content

    def __repr__(self) -> str:
        return f"Agent('{self.agent_id}', role='{self.role}')"

### Blackboard

The `Blackboard` provides a read-only aggregated view of all shared memory. Agents use it to see the current state of the collaborative work without querying each namespace separately.

In [ ]:
class Blackboard:
    """Aggregates shared memory into a unified read-only view."""

    def __init__(self, shared_memory: SharedMemoryPool):
        self.shared_memory = shared_memory

    def get_summary(self, agent_id: str) -> str:
        """Build a text summary of everything the agent can see."""
        entries = self.shared_memory.read_all_accessible(agent_id)
        if not entries:
            return "The blackboard is empty. No entries posted yet."

        sections: dict[str, list[str]] = {}
        for entry in entries:
            sections.setdefault(entry.namespace, []).append(
                f"  [{entry.key}] (v{entry.version}, by {entry.author}): "
                f"{entry.value[:120]}"
            )

        lines = ["Current Blackboard State:", ""]
        for ns in sorted(sections):
            lines.append(f"=== {ns}/ ===")
            lines.extend(sections[ns])
            lines.append("")

        return "\n".join(lines)

## Example Run

We'll set up a three-agent team that collaborates on a programming task:

- **Researcher**: Gathers requirements. Writes to `research/`, reads everything.
- **Coder**: Writes code. Writes to `code/`, reads everything.
- **Reviewer**: Reviews code. Writes to `review/`, reads everything.

Each agent writes to its own namespace but reads from all namespaces. This prevents accidental overwrites while allowing full visibility.

### Configure Permissions and Create Agents

First we set up the access control rules and create the agent team.

In [ ]:
# Create the access control layer
acl = AccessControl()

# Researcher: writes to research/, reads everything
acl.grant("researcher", "research", Permission.READ_WRITE)
acl.grant("researcher", "code", Permission.READ)
acl.grant("researcher", "review", Permission.READ)

# Coder: writes to code/, reads everything
acl.grant("coder", "research", Permission.READ)
acl.grant("coder", "code", Permission.READ_WRITE)
acl.grant("coder", "review", Permission.READ)

# Reviewer: writes to review/, reads everything
acl.grant("reviewer", "research", Permission.READ)
acl.grant("reviewer", "code", Permission.READ)
acl.grant("reviewer", "review", Permission.READ_WRITE)

print("Permission matrix:")
print(acl.describe())

Now we create the shared memory pool, blackboard, and the three agents.
Each agent gets a role-specific system prompt that guides its behavior.
The shared memory pool connects all agents through the access control layer we defined above.

In [ ]:
# Create the shared memory pool
shared_memory = SharedMemoryPool(acl, conflict_strategy="last_write_wins")

# Create the blackboard view
blackboard = Blackboard(shared_memory)

# Create the agents
researcher = Agent(
    agent_id="researcher",
    role="Research Specialist",
    system_prompt=(
        "You are a research specialist. Analyze requirements and list "
        "the key technical details. Be specific and concise. "
        "Write no more than 8 bullet points."
    ),
    shared_memory=shared_memory,
)

coder = Agent(
    agent_id="coder",
    role="Software Engineer",
    system_prompt=(
        "You are a software engineer. Write clean, working Python code "
        "based on the research findings in shared memory. "
        "Keep code short and well-commented. Include example usage."
    ),
    shared_memory=shared_memory,
)

reviewer = Agent(
    agent_id="reviewer",
    role="Code Reviewer",
    system_prompt=(
        "You are a code reviewer. Review the code for correctness, "
        "clarity, and edge cases. Be constructive. Point out specific "
        "issues and suggest fixes. Keep feedback to 5 items or fewer."
    ),
    shared_memory=shared_memory,
)

print(f"\nAgents created: {researcher}, {coder}, {reviewer}")

### Run the Collaborative Task

The three agents work on a task in sequence. Each agent reads the previous agents' contributions from shared memory, does its work, and posts results back.

The task: build a Python function that converts temperatures between Celsius, Fahrenheit, and Kelvin.

In [ ]:
task = "Build a Python function that converts temperatures between Celsius, Fahrenheit, and Kelvin."

print("=" * 60)
print(f"TASK: {task}")
print("=" * 60)

# Step 1: Researcher gathers requirements
print("\n--- Step 1: Researcher gathers requirements ---\n")
research_output = researcher.run(
    f"Research the requirements for this task: {task}\n\n"
    "List the conversion formulas, edge cases, and design considerations."
)
print(f"Researcher:\n{research_output}\n")

# Researcher posts findings to shared memory
researcher.write_shared("research", "requirements", research_output)
researcher.think("Posted requirements to research/requirements")  # private note

# Step 2: Coder writes the implementation
print("\n--- Step 2: Coder writes implementation ---\n")
coder_output = coder.run(
    "Write a Python implementation based on the research requirements "
    "in shared memory. Include the function and example calls.",
    context_namespaces=["research"],
)
print(f"Coder:\n{coder_output}\n")

# Coder posts code to shared memory
coder.write_shared("code", "implementation", coder_output)

# Step 3: Reviewer checks the work
print("\n--- Step 3: Reviewer checks the code ---\n")
reviewer_output = reviewer.run(
    "Review the code in shared memory against the research requirements. "
    "List any bugs, missing edge cases, or improvements.",
    context_namespaces=["research", "code"],
)
print(f"Reviewer:\n{reviewer_output}")

# Reviewer posts feedback to shared memory
reviewer.write_shared("review", "feedback", reviewer_output)

### Inspect the Shared Memory State

After the agents finish, we can view the full blackboard and the operation history. The history is an audit trail showing what each agent contributed and when.

In [ ]:
# View the full blackboard
print("=" * 60)
print("FINAL BLACKBOARD STATE")
print("=" * 60)
print(blackboard.get_summary("reviewer"))  # reviewer can see everything

# View the audit log
print("\n--- Operation History ---")
for op in shared_memory.get_history():
    print(f"  {op['action'].upper()} {op['key']} (v{op['version']}) by {op['agent']}")

### Private vs. Shared Memory

The researcher added a private scratchpad note in the previous step. Let's confirm that private notes stay private and don't appear in shared memory.

In [ ]:
# The researcher's private scratchpad
print("Researcher's private scratchpad:")
for note in researcher.scratchpad:
    print(f"  - {note}")

# The coder and reviewer have empty scratchpads
print(f"\nCoder's scratchpad: {coder.scratchpad}")
print(f"Reviewer's scratchpad: {reviewer.scratchpad}")

# Shared memory has only what was explicitly written
print(f"\nEntries in shared memory: {len(shared_memory._store)}")
print("Private scratchpad notes do NOT appear in shared memory.")

### Permission Enforcement

Each agent can only write to its own namespace. Writing to another agent's namespace raises a `PermissionError`. This is how access control prevents unauthorized changes.

In [ ]:
# The researcher cannot write to the code/ namespace
try:
    researcher.write_shared("code", "sneaky_write", "This should fail")
    print("ERROR: Write should have been blocked!")
except PermissionError as e:
    print(f"Blocked (expected): {e}")

# The coder cannot write to the research/ namespace
try:
    coder.write_shared("research", "sneaky_write", "This should also fail")
    print("ERROR: Write should have been blocked!")
except PermissionError as e:
    print(f"Blocked (expected): {e}")

print("\nPermission enforcement works correctly.")

### Conflict Resolution

When two agents write to the same key, we need a strategy. Let's compare last-write-wins and version-check approaches.

**Last-write-wins** is the default. The most recent write overwrites the old value. This is fast but can lose data.

**Version check** (optimistic locking) requires the writer to know the current version. If someone else updated the entry since you last read it, the write fails. This catches conflicts instead of silently discarding data.

In [ ]:
# --- Last-write-wins demo ---
print("=== Last-Write-Wins Strategy ===\n")

demo_acl = AccessControl()
demo_acl.grant("agent_a", "shared", Permission.READ_WRITE)
demo_acl.grant("agent_b", "shared", Permission.READ_WRITE)

lww_memory = SharedMemoryPool(demo_acl, conflict_strategy="last_write_wins")

entry_a = lww_memory.write("agent_a", "shared", "status", "Task is 50% complete")
print(f"Agent A writes: v{entry_a.version} -> '{entry_a.value}'")

entry_b = lww_memory.write("agent_b", "shared", "status", "Task is 75% complete")
print(f"Agent B writes: v{entry_b.version} -> '{entry_b.value}'")

result = lww_memory.read("agent_a", "shared", "status")
print(f"Final value: '{result[0].value}' (by {result[0].author}, v{result[0].version})")
print("Agent B's write overwrote Agent A's value.\n")

# --- Version check demo ---
print("=== Version Check Strategy ===\n")

vc_memory = SharedMemoryPool(demo_acl, conflict_strategy="version_check")

# Agent A writes the initial value
entry = vc_memory.write("agent_a", "shared", "status", "Draft v1")
print(f"Agent A writes: v{entry.version} -> '{entry.value}'")

# Agent B reads v1, then writes with expected_version=1
entry_b = vc_memory.write(
    "agent_b", "shared", "status", "Revised by B",
    expected_version=1,
)
print(f"Agent B writes (expected v1): v{entry_b.version} -> '{entry_b.value}'")

# Agent A tries to write with stale version (expected v1, but it's now v2)
try:
    vc_memory.write(
        "agent_a", "shared", "status", "Revised by A",
        expected_version=1,  # stale: actual version is now 2
    )
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Conflict detected (expected): {e}")

print("\nVersion check catches stale writes instead of silently overwriting.")

## Tradeoffs

### When Multi-Agent Shared Memory Works Well

- **Collaborative workflows with clear roles.** When agents have distinct responsibilities (research, code, review), shared memory provides clean hand-off points. Each agent writes to its own namespace and reads from others.
- **Tasks that benefit from iterative refinement.** The blackboard pattern lets agents build on each other's work. A reviewer's feedback can trigger a coder's revision. Each pass improves the result.
- **Audit and debugging.** The operation history provides a clear record of what each agent contributed. You can trace how the final result was assembled from individual contributions.
- **Parallel agent execution.** Thread-safe shared memory lets multiple agents work at the same time. The lock prevents data corruption from concurrent writes.

### When It Breaks Down

- **High-frequency concurrent writes.** If many agents write to the same keys rapidly, the lock becomes a bottleneck. Last-write-wins loses data. Version checking triggers frequent retries.
- **Large memory stores.** As shared memory grows, agents read more context on each turn. This increases token usage and cost. You may need retrieval-based access (searching for relevant entries) instead of reading everything.
- **Tightly coupled agents.** If agents depend on each other's outputs in real-time (like a back-and-forth conversation), shared memory adds overhead. Direct message passing may be faster for tightly coupled workflows.
- **Security-sensitive data.** The permission model shown here runs in application code. A bug could leak private scratchpad content into shared memory. Production systems need stronger isolation (for example, separate processes or encryption).

## Further Reading

- Nii, H. P. (1986). ["Blackboard Systems."](https://doi.org/10.1609/aimag.v7i2.537) *AI Magazine*, 7(2), 38-53. The foundational paper on blackboard architectures for multi-agent collaboration.

- [LangGraph Multi-Agent Patterns](https://langchain-ai.github.io/langgraph/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): LangGraph's approach to multi-agent state management with shared state graphs.

- [CrewAI Shared Memory](https://docs.crewai.com/concepts/memory?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): CrewAI's implementation of shared and per-agent memory in collaborative agent teams.

- [AutoGen Multi-Agent Conversations](https://microsoft.github.io/autogen/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Microsoft's framework for multi-agent conversations with structured message protocols.

---

*← Previous: [21: Cross-Session Memory](../21_cross_session_memory/) · Next: [23: Memory with Tools](../23_memory_with_tools/) →*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: New specialist agent
Add a 'debugger' `Agent` to the system with its own namespace and read-only access to the 'researcher' and 'coder' namespaces via `AccessControl`. Run a three-agent workflow and verify the debugger can read but not write to other namespaces.

### Challenge 2: Conflict rate measurement
Have 3 agents write to the same namespace key in rapid succession using version-check conflict resolution in `SharedMemoryPool`. Run 20 write attempts and count how many trigger a version conflict. Report the conflict rate and the final value after all writes.

### Challenge 3: Priority-based conflict resolution
Replace last-write-wins with an importance-scored resolution strategy. Each write includes a priority score; the `SharedMemoryPool` keeps the higher-priority value on conflict. Test with agents writing contradictory facts at different priorities. This draws on the importance scoring ideas from 14 Memory Consolidation.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--22-multi-agent-shared-memory--multi-agent-shared-memory)